<!--
File: notebooks_v2/04_application_integration_v2.ipynb
What does this file do?
    Demonstrates how the trained model is integrated into an application payload and the Go/React web app.
Methods/functions this file contains:
    Notebook cells for running FinanceAssistantV2 with an app-style payload and inspecting insights.
Date and Day of last modification:
    2026-05-21, Thursday.
-->

# 04 - Application Integration V2

This notebook shows how the trained model can be used by an application backend.

Web app folder: `web/`
Model entrypoint: `assistant_v2.finance_assistant.FinanceAssistantV2`

## Integration Design

The app sends a backend-shaped payload. The middleware normalizes it into the model schema and removes sensitive identity fields from model input.

Final response includes:

- forecast
- budget risk
- anomaly
- category summary
- shared summary
- shared settlement risk
- user-facing insights

In [ ]:
from pathlib import Path
import pandas as pd
from assistant_v2.finance_assistant import FinanceAssistantV2

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks_v2' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'output_v2'
users = pd.read_csv(DATA_DIR / 'users.csv')
expenses = pd.read_csv(DATA_DIR / 'expenses.csv', low_memory=False)
income = pd.read_csv(DATA_DIR / 'income_events.csv')
budgets = pd.read_csv(DATA_DIR / 'budgets.csv')
shared = pd.read_csv(DATA_DIR / 'shared_expenses.csv')

shared_user_id = expenses[expenses['is_shared'].astype(str).str.lower().eq('true')].iloc[0]['user_id']
user_expenses = expenses[expenses['user_id'] == shared_user_id].tail(500)
payload = {
    'user': users[users['user_id'] == shared_user_id].iloc[0].to_dict(),
    'expenses': user_expenses.to_dict('records'),
    'income': income[income['user_id'] == shared_user_id].tail(12).to_dict('records'),
    'budgets': budgets[budgets['user_id'] == shared_user_id].to_dict('records'),
    'shared_expenses': shared[shared['expense_id'].isin(user_expenses['expense_id'])].tail(50).to_dict('records'),
}

assistant = FinanceAssistantV2(PROJECT_ROOT / 'models_v2_nepal')
response = assistant.generate_insights(payload)
response.keys()

In [ ]:
response['summary']

In [ ]:
pd.DataFrame(response['insights'])[['type', 'severity', 'title', 'message', 'action']]

## Web Application

The web app is inside `web/`.

Run from the `web` folder:

```powershell
$env:GOTELEMETRY='off'
go run ./server
```

Then open `http://localhost:8080`.